# Ecuaciones diferenciales de primer orden

## Bibliotecas

In [ ]:
import sympy as sp
import numpy as np
import plotly.graph_objects as go
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

## Funciones generales

In [ ]:
def analizar_ecuacion_diferencial(ecuacion, y):
    print("Ecuación diferencial planteada:")
    display(ecuacion)

    clasificacion = sp.classify_ode(ecuacion, y)

    # Mostrar las caracteristicas de la ecuación diferencial
    print("Clasificaciones de la ecuación diferencial:")

    es_lineal = any('linear' in c for c in clasificacion)
    print(f"Es lineal: {es_lineal}")

    es_ordinaria = any('ordinary' in c for c in clasificacion)
    print(f"Es ordinaria: {es_ordinaria}")

    orden = sp.ode_order(ecuacion, y)
    print(f"Orden: {orden}")

    print("Posibles métodos de solución:")
    for metodo in clasificacion:
        print(f"- {metodo}")

In [ ]:
def solucion_particular(solucion_general, y0):
    expresion = solucion_general.rhs
    C = next(iter(expresion.free_symbols - {x}))

    condicion = sp.Eq(expresion.subs(x, 0), y0)
    soluciones_C = sp.solve(condicion, C, check=False)

    if not soluciones_C:
        condicion = sp.powdenest(condicion, force=True)
        soluciones_C = sp.solve(condicion, C, check=False)

    if not soluciones_C:
        raise ValueError("No se pudo determinar la constante de integración.")

    return sp.Eq(
        y,
        sp.simplify(expresion.subs(C, soluciones_C[0]))
    )

In [ ]:
def graficar_campo_ecuacion(ecuacion, x_min=0, x_max=15, y_min=-15, y_max=15):
    num_points = 30
    
    # Crear una malla de puntos en el plano (x, y)
    x_vals = np.linspace(x_min, x_max, num_points)
    y_vals = np.linspace(y_min, y_max, num_points)
    X, Y = np.meshgrid(x_vals, y_vals)

    # Calcular las derivadas en cada punto de la malla
    dydx = sp.lambdify((x, y), ecuacion.rhs)  # Convertir la ecuación a una función numérica
    U = np.ones_like(X)
    V = dydx(X, Y)  # Componente y del vector

    # Normalizar los vectores para que tengan longitud 1
    magnitud = np.sqrt(U**2 + V**2)
    U_norm = U / magnitud
    V_norm = V / magnitud

    angulo = np.arctan2(V_norm, U_norm)

    # Definir escala de los vectores para que se vean bien en la gráfica
    fig = plt.figure(figsize=(9, 5))
    normalizacion_absoluta = mcolors.Normalize(vmin=0, vmax=np.pi)
    plt.quiver(X, Y, U_norm, V_norm, angulo, cmap='brg', norm=normalizacion_absoluta, pivot='mid', scale=50)
    plt.title("Campo de direcciones de la ecuación diferencial")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['bottom'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.close(fig)

    return fig

In [ ]:
def convertir_raices_reales(expr):
    return expr.replace(
        lambda e: (
            isinstance(e, sp.Pow)
            and e.exp.is_Rational
            and e.exp.q % 2 == 1
        ),
        lambda e: sp.real_root(e.base, e.exp.q) ** e.exp.p
    )

def graficar_solucion_particular(
    solucion_particular,
    x_min=0,
    x_max=15,
    num_puntos=200
):
    
    expresion = convertir_raices_reales(solucion_particular.rhs)

    # Convertir a función numérica
    f = sp.lambdify(x, expresion, 'numpy')

    x_vals = np.linspace(x_min, x_max, num_puntos)
    y_vals = []
    with np.errstate(all='ignore'):
        for x_val in x_vals:
            y_ = f(x_val)
            y_vals.append(y_)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=x_vals,
        y=y_vals,
        mode='lines',
        name='Solución particular'
    ))

    fig.update_layout(
        title='Solución particular',
        xaxis_title='x',
        yaxis_title='y',
        template='plotly_white'
    )

    return fig

## Definir ecuacion

In [ ]:
x = sp.Symbol('x')
y = sp.Function('y')(x)
derivada = sp.Derivative(y, x)

In [ ]:
ecuacion = sp.Eq(derivada, -9 * x + 3 * y)
ecuacion = sp.Eq(derivada, (2 * x) / (3 * y ** 2))

In [ ]:
analizar_ecuacion_diferencial(ecuacion, y)

## Solución general

In [ ]:
solucion_general = sp.dsolve(ecuacion, y)
if isinstance(solucion_general, list):
    solucion_general = solucion_general[0]

print("La solución general de la ecuación diferencial es:")
display(solucion_general)

In [ ]:
grafica_campo = graficar_campo_ecuacion(ecuacion, x_min=0, x_max=10, y_min=-5, y_max=10)
display(grafica_campo)

## Solucion particular

In [ ]:
y_0 = -2
sol_particular = solucion_particular(solucion_general, y0=y_0)

print(f"Solución particular con y(0) = {y_0}:")
display(sol_particular)

In [ ]:
grafica = graficar_solucion_particular(sol_particular, x_min=0, x_max=10)
grafica.show()